In [3]:
!python train.py


usage: train.py [-h] --data DATA [--steps STEPS] [--batch BATCH] [--lr LR]
                [--seed SEED] [--out OUT] [--log_every LOG_EVERY]
train.py: error: the following arguments are required: --data


In [4]:
!python train.py --data train_corpus.txt

corpus: 7,318,592 bytes -> 7,318,592 tokens (vocab 256)
model: 1,339,840 params
step     1  loss 5.6475  (478 ms/step)
step   100  loss 2.8522  (134 ms/step)
step   200  loss 2.2753  (132 ms/step)
step   300  loss 2.2126  (132 ms/step)
step   400  loss 2.1896  (130 ms/step)
step   500  loss 2.1897  (130 ms/step)
step   600  loss 2.1240  (129 ms/step)
step   700  loss 2.0959  (129 ms/step)
step   800  loss 2.0686  (129 ms/step)
step   900  loss 2.0983  (129 ms/step)
step  1000  loss 2.0164  (130 ms/step)
step  1100  loss 1.9847  (130 ms/step)
step  1200  loss 1.9508  (129 ms/step)
step  1300  loss 1.8913  (130 ms/step)
step  1400  loss 1.8323  (129 ms/step)
step  1500  loss 1.8090  (130 ms/step)
step  1600  loss 1.7905  (130 ms/step)
step  1700  loss 1.7931  (130 ms/step)
step  1800  loss 1.7434  (130 ms/step)
step  1900  loss 1.7505  (130 ms/step)
step  2000  loss 1.7315  (130 ms/step)
saved ckpt.pt  (260s total)


In [5]:
!python evaluate.py --text_file dev_eval.txt


{"bpb": 2.3718, "n_params": 1339840, "steps": 2000, "tokens_in_eval": 159225, "tokens_scored": 159224}


In [16]:
%%writefile model.py
import torch
import torch.nn as nn
from torch.nn import functional as F

class Head(nn.Module):
    def __init__(self, head_size, n_embd, block_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (C ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, n_embd, block_size, dropout):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size, n_embd, block_size, dropout) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size, n_embd, block_size, dropout)
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size=256, block_size=256, n_embd=192, n_head=4, n_layer=4, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)


        self.lm_head.weight = self.token_embedding_table.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits_view = logits.view(B * T, C)
            targets_view = targets.view(B * T)
            loss = F.cross_entropy(logits_view, targets_view)

        return logits, loss


GPT = GPTLanguageModel
class Config:
    pass

Overwriting model.py


In [17]:
%%writefile train.py
import math
import torch
from model import GPTLanguageModel
from tokenizer import ByteTokenizer


batch_size = 32
block_size = 256
max_iters = 2000
learning_rate = 3e-4
weight_decay = 0.1
eval_interval = 200
device = 'cuda' if torch.cuda.is_available() else 'cpu'

with open('train_corpus.txt', 'r', encoding='utf-8') as f:
    text = f.read()

tokenizer = ByteTokenizer()
data = torch.tensor(tokenizer.encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data_source = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_source) - block_size, (batch_size,))
    x = torch.stack([data_source[i:i+block_size] for i in ix])
    y = torch.stack([data_source[i+1:i+1+block_size] for i in ix])
    return x.to(device), y.to(device)

model = GPTLanguageModel(vocab_size=256, block_size=block_size, n_embd=192, n_head=4, n_layer=4, dropout=0.1)
model.to(device)

print(f"Total Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

def get_lr(it):
    warmup_iters = 100
    if it < warmup_iters:
        return learning_rate * it / warmup_iters
    if it > max_iters:
        return learning_rate * 0.1
    decay_ratio = (it - warmup_iters) / (max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return learning_rate * (0.1 + 0.9 * coeff)

for iter in range(max_iters):
    lr = get_lr(iter)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if iter % eval_interval == 0 or iter == max_iters - 1:
        print(f"Step {iter}: train loss {loss.item():.4f}, lr {lr:.5f}")


checkpoint = {
    'model_state_dict': model.state_dict(),
    'config': {
        'vocab_size': 256,
        'block_size': block_size,
        'n_embd': 192,
        'n_head': 4,
        'n_layer': 4,
        'dropout': 0.1
    }
}
torch.save(checkpoint, 'ckpt.pt')
print("Training completed and checkpoint saved with config dictionary as ckpt.pt")

Overwriting train.py


In [18]:
!python train.py --data train_corpus.txt

Total Parameters: 1.88M
Step 0: train loss 5.6827, lr 0.00000
Step 200: train loss 2.1924, lr 0.00030
Step 400: train loss 2.1518, lr 0.00028
Step 600: train loss 2.0531, lr 0.00026
Step 800: train loss 2.1921, lr 0.00022
Step 1000: train loss 2.0861, lr 0.00018
Step 1200: train loss 2.1358, lr 0.00013
Step 1400: train loss 2.1490, lr 0.00009
Step 1600: train loss 2.0043, lr 0.00006
Step 1800: train loss 1.8384, lr 0.00004
Step 1999: train loss 1.7164, lr 0.00003
Training completed and checkpoint saved with config dictionary as ckpt.pt


In [22]:
%%writefile evaluate.py
import argparse
import torch
import torch.nn.functional as F
from model import GPTLanguageModel
from tokenizer import ByteTokenizer

def load_model(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    cfg = checkpoint['config']

    model = GPTLanguageModel(
        vocab_size=cfg['vocab_size'],
        block_size=cfg['block_size'],
        n_embd=cfg['n_embd'],
        n_head=cfg['n_head'],
        n_layer=cfg['n_layer'],
        dropout=cfg['dropout']
    )
    model.load_state_dict(checkpoint['model_state_dict'])
    return model, cfg

def main():
    parser = argparse.ArgumentParser(description="Evaluate GPT language model.")
    parser.add_argument('--checkpoint', type=str, default='ckpt.pt', help='Path to model checkpoint')
    parser.add_argument('--text_file', type=str, required=True, help='Path to evaluation text file')
    args = parser.parse_args()

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Loading model from {args.checkpoint} onto {device}...")

    model, cfg = load_model(args.checkpoint)
    model.to(device)
    model.eval()

    print(f"Model loaded successfully with {sum(p.numel() for p in model.parameters()):,d} parameters.")

    with open(args.text_file, 'r', encoding='utf-8') as f:
        text = f.read()

    tokenizer = ByteTokenizer()
    tokens = tokenizer.encode(text)
    data = torch.tensor(tokens, dtype=torch.long, device=device)

    block_size = cfg['block_size']
    total_loss = 0.0
    tokens_scored = 0

    print(f"Evaluating on {args.text_file}...")
    with torch.no_grad():
        for i in range(0, len(data) - block_size, block_size):
            x = data[i:i+block_size].unsqueeze(0)
            y = data[i+1:i+1+block_size].unsqueeze(0)
            if x.size(1) < block_size:
                break
            logits, loss = model(x, y)
            total_loss += loss.item() * x.size(1)
            tokens_scored += x.size(1)

    avg_loss = total_loss / tokens_scored if tokens_scored > 0 else 0.0
    bpb = avg_loss / math.log(2) if 'math' in globals() else avg_loss / 0.693147

    print(f"\nEvaluation Results:")
    print(f"Tokens Scored: {tokens_scored}")
    print(f"Average Loss: {avg_loss:.4f}")
    print(f"Bits-Per-Byte (BPB): {bpb:.4f}")

if __name__ == '__main__':
    import math
    main()

Overwriting evaluate.py


In [23]:
!python evaluate.py --text_file dev_eval.txt

Loading model from ckpt.pt onto cuda...
Model loaded successfully with 1,875,840 parameters.
Evaluating on dev_eval.txt...

Evaluation Results:
Tokens Scored: 158976
Average Loss: 1.7944
Bits-Per-Byte (BPB): 2.5887


In [24]:
import zipfile
from google.colab import files

files_to_zip = ['model.py', 'train.py', 'evaluate.py', 'tokenizer.py', 'ckpt.pt']
zip_filename = 'assignment_submission.zip'

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        zipf.write(file)

files.download(zip_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>